# Module 14: MCP & Tool Design

**Goal:** Build tools agents can actually use correctly.

**What you'll learn:**
- Model Context Protocol (MCP) architecture and primitives
- Tool descriptions as routing signals
- Schema design to eliminate ambiguity
- Error handling at protocol and business-logic levels
- Agent-Computer Interface (ACI) design principles

**Prerequisites:** Module 07 (agentic workflows)

---

## 0. Setup

In [ ]:
%pip install -q mcp

In [ ]:
print("✅ Setup complete")
print("Note: MCP servers run as separate processes.")
print("This notebook demonstrates tool design principles.")

---
## 1. Tool Description Quality

The description is the **primary routing signal** — the model reads it to decide whether to call this tool.

In [ ]:
# BAD: Describes WHAT, not WHEN
bad_tool = {
    "name": "search_documents",
    "description": "Searches documents and returns results."
}

# GOOD: Tells WHEN to call this tool
good_tool = {
    "name": "search_documents",
    "description": """Search the internal knowledge base for company documents, policies, and procedures.

Use when the user asks about internal company information, policies, HR procedures,
or any topic that might be documented internally. Returns top 3 matching excerpts.

Do NOT use for general web search (use web_search instead) or real-time data."""
}

print("BAD description:")
print(f"  {bad_tool['description']}")
print("\nGOOD description:")
print(f"  {good_tool['description'][:100]}...")

---
## 2. Schema Design — Reduce Ambiguity

Use constrained types and format examples to prevent hallucinated arguments.

In [ ]:
# BAD: Free-form string allows anything
bad_schema = {
    "name": "create_event",
    "parameters": {
        "title": {"type": "string"},
        "time": {"type": "string"}  # "next Tuesday at 3pm"??
    }
}

# GOOD: Constrained types + format examples
good_schema = {
    "name": "create_event",
    "parameters": {
        "title": {"type": "string", "description": "Event title"},
        "start_datetime": {
            "type": "string",
            "description": "ISO 8601 format, e.g. '2026-07-17T14:30:00'"
        },
        "calendar": {
            "type": "string",
            "enum": ["personal", "work", "team"]
        }
    }
}

print("Schema checklist:")
print("  ✓ Use enum for fixed-choice parameters")
print("  ✓ Include format examples in descriptions")
print("  ✓ Only mark truly required fields as required")
print("  ✓ Add description to every parameter")

---
## 3. Error Handling — Two Layers

MCP distinguishes protocol errors (JSON-RPC level) from tool execution errors (business logic).

In [ ]:
# Good error messages include:
# 1. What failed and why (user-understandable)
# 2. What the model or user can do to fix it
# 3. Never expose stack traces or internal URLs

error_examples = {
    "invalid_address": "Invalid email address: 'foo'. Please provide a valid email (e.g. name@domain.com).",
    "rate_limit": "Rate limit reached. Wait 30s and retry.",
    "not_found": "User 'bob' not found. Did you mean 'bobsmith'?",
}

for error_type, message in error_examples.items():
    print(f"  {error_type}: {message}")

---
## 4. Agent-Computer Interface (ACI)

Anthropic found that **tool design matters more than prompt design**. They spent more time optimizing tools than the overall agent prompt.

In [ ]:
# ACI Principles:
aci_checklist = [
    "Put yourself in the model's shoes — is it obvious how to use this tool?",
    "Include example usage, edge cases, and format requirements",
    "Change parameter names to make mistakes harder (poka-yoke)",
    "Use absolute paths, explicit formats, constrained types",
    "Test with many examples — watch what mistakes the model makes",
]

print("ACI Checklist:")
for i, item in enumerate(aci_checklist, 1):
    print(f"  {i}. {item}")

print("\nExample: Absolute paths vs relative paths")
print("  BAD:  edit_file(path='src/main.py')  # breaks when agent cd's")
print("  GOOD: edit_file(absolute_path='/home/user/project/src/main.py')  # always works")

---
## Summary

| Principle | Why It Matters |
|-----------|---------------|
| **Description = routing signal** | Model decides tool based on description |
| **Constrained schemas** | Prevents hallucinated arguments |
| **Actionable errors** | Agent can self-repair on failure |
| **ACI > prompt engineering** | Tool design has higher ROI than prompt tweaks |

**Good tools make agents more capable. Bad tools make them unpredictable.**

---
## 🧪 Exercises

1. **Description Rewrite**: Take 3 tools with bad descriptions. Rewrite them with trigger conditions. Does routing improve?

2. **Schema Ambiguity**: Build a scheduling tool with free-form `time` param. Call it 20 times. How often does the agent pass invalid formats? Now add strict ISO 8601.

3. **ACI Audit**: Review any tool you've built. Apply the ACI checklist. What would you change?